# Part 2.2 — Cybersecurity: Synthetic Traffic Generation with CICIDS 2017

This notebook builds a **tabular GAN** for the **CICIDS 2017** data, with:
- **multi-day exploration** to understand class balance across the dataset
- **GAN training on Wednesday only**, aligned with the assignment brief
- **Google Drive support**, so it runs cleanly in Colab

## Notebook goals
1. Explore CICIDS across multiple days.
2. Understand class balance and attack diversity across days.
3. Select the Wednesday file for modelling.
4. Clean, scale, and prepare the Wednesday feature space.
5. Train an **MLP-based GAN** on tabular network traffic data.
6. Compare real vs generated traffic using:
   - training loss curves
   - PCA visualisation
   - t-SNE visualisation
   - selected feature-distribution comparisons
7. Save report-ready plots and CSV outputs.

## Assignment alignment
- **Explore multiple daily files** to understand the broader dataset.
- **Train the GAN on benign and DoS attacks from the Wednesday file only**.

In [ ]:
# ============================================================
# CELL 1 — IMPORTS, REPRODUCIBILITY, AND VISUAL STYLE
# ============================================================

from pathlib import Path
import random
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import pairwise_distances

warnings.filterwarnings("ignore")

SEED = 42

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

COLOR_REAL = "#7A1F1F"
COLOR_FAKE = "#C97B63"
COLOR_ALT = "#6C8A3B"
COLOR_GRID = "#D9D2C3"
COLOR_TEXT = "#2F2A24"
COLOR_ATTACK = "#8A4F7D"

plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams["axes.facecolor"] = "#FAF7F2"
plt.rcParams["figure.facecolor"] = "#FFFFFF"
plt.rcParams["axes.edgecolor"] = "#B9AD99"
plt.rcParams["grid.color"] = COLOR_GRID
plt.rcParams["axes.labelcolor"] = COLOR_TEXT
plt.rcParams["xtick.color"] = COLOR_TEXT
plt.rcParams["ytick.color"] = COLOR_TEXT
plt.rcParams["text.color"] = COLOR_TEXT
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.grid"] = True

In [ ]:
# ============================================================
# CELL 2 — GOOGLE DRIVE MOUNT AND DATA DIRECTORY SETUP
# ============================================================
"""
Mount Google Drive and set the folder containing the CICIDS CSV files.

IMPORTANT:
1. Mount Drive.
2. Change DATA_DIR below to your actual folder.
3. Keep all required CSV files inside that folder.
"""

from google.colab import drive
drive.mount('/content/drive')

# ------------------------------------------------
# CHANGE THIS FOLDER PATH TO YOUR OWN DRIVE FOLDER
# ------------------------------------------------
DATA_DIR = Path("/content/drive/MyDrive/CICIDS_2017")

OUTPUT_DIR = Path("/content/drive/MyDrive/CICIDS_2017_outputs") / "cicids"
CSV_DIR = OUTPUT_DIR / "csv"
PLOTS_DIR = OUTPUT_DIR / "plots"

for folder in [OUTPUT_DIR, CSV_DIR, PLOTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"DATA_DIR: {DATA_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"CSV_DIR: {CSV_DIR}")
print(f"PLOTS_DIR: {PLOTS_DIR}")

def save_plot(filename: str) -> None:
    filepath = PLOTS_DIR / filename
    plt.tight_layout()
    plt.savefig(filepath, dpi=300, bbox_inches="tight")
    print(f"Saved plot: {filepath}")

In [ ]:
# ============================================================
# CELL 3 — DEFINE FILE PATHS AND CHECK THEY EXIST
# ============================================================

DATA_FILES = {
    "Monday": DATA_DIR / "Monday-WorkingHours.pcap_ISCX.csv",
    "Tuesday": DATA_DIR / "Tuesday-WorkingHours.pcap_ISCX.csv",
    "Wednesday": DATA_DIR / "Wednesday-workingHours.pcap_ISCX.csv",
    "Thursday": DATA_DIR / "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "Friday": DATA_DIR / "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
}

print("Files to process:")
for day, path in DATA_FILES.items():
    print(f"{day}: {path} | exists={path.exists()}")

missing_files = [str(path) for path in DATA_FILES.values() if not Path(path).exists()]
if missing_files:
    raise FileNotFoundError(
        "These files are missing from the selected Google Drive folder:\n" + "\n".join(missing_files)
    )

DATA_PATH = DATA_FILES["Wednesday"]
print(f"\nWednesday training file: {DATA_PATH}")

## Multi-Day Exploration

This section satisfies the brief’s requirement to understand the **broader CICIDS dataset** across multiple days.

To keep the notebook memory-safe in Colab:
- files are processed **in chunks**
- only the **Label** column is analysed in the multi-day stage
- the GAN itself is later trained on **Wednesday only**

In [ ]:
# ============================================================
# CELL 4 — MEMORY-SAFE LABEL EXTRACTION (CHUNK PROCESSING)
# ============================================================

def extract_label_counts(filepath: Path, chunksize: int = 200000):
    """
    Extract label counts from a large CSV file using chunk processing.
    """
    label_counts = {}
    total_rows = 0

    for chunk in pd.read_csv(filepath, chunksize=chunksize):
        chunk.columns = [col.strip() for col in chunk.columns]
        counts = chunk["Label"].value_counts()

        for label, count in counts.items():
            label_counts[label] = label_counts.get(label, 0) + int(count)

        total_rows += len(chunk)

    return label_counts, total_rows

all_day_summaries = []

for day, path in DATA_FILES.items():
    print(f"\nProcessing {day}...")
    label_counts, total_rows = extract_label_counts(path)

    for label, count in label_counts.items():
        all_day_summaries.append({
            "day": day,
            "label": label,
            "count": count,
            "total_rows_day": total_rows
        })

multi_day_df = pd.DataFrame(all_day_summaries)
multi_day_df.to_csv(CSV_DIR / "00_multiday_label_distribution.csv", index=False)

display(multi_day_df.head())

In [ ]:
# ============================================================
# CELL 5 — MULTI-DAY PIVOT SUMMARY TABLE
# ============================================================

pivot_df = multi_day_df.pivot_table(
    index="day",
    columns="label",
    values="count",
    aggfunc="sum",
    fill_value=0
)

pivot_df["total_samples"] = pivot_df.sum(axis=1)

pivot_df.to_csv(CSV_DIR / "01_multiday_pivot_summary.csv")
display(pivot_df)

In [ ]:
# ============================================================
# CELL 6 — VISUALISE CLASS DISTRIBUTION ACROSS DAYS
# ============================================================

pivot_plot = pivot_df.drop(columns=["total_samples"], errors="ignore")

pivot_plot.plot(
    kind="bar",
    stacked=True,
    figsize=(12, 6),
    colormap="tab20"
)

plt.title("CICIDS Multi-Day Class Distribution")
plt.xlabel("Day")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
save_plot("00_multiday_class_distribution.png")
plt.show()

### Why Wednesday was selected

This exploration is intended to justify the modelling choice:

- **Monday** is mostly benign traffic
- **Tuesday** includes other attack families
- **Thursday** includes web attacks
- **Friday** includes port scanning
- **Wednesday** contains **BENIGN** plus multiple **DoS attack classes**, making it the strongest match to the assignment objective

So the GAN training stage below focuses on **Wednesday only**.

In [ ]:
# ============================================================
# CELL 7 — LOAD THE WEDNESDAY DATA AND STANDARDISE COLUMNS
# ============================================================

df_raw = pd.read_csv(DATA_PATH)
df_raw.columns = [col.strip() for col in df_raw.columns]

print(f"Raw Wednesday shape: {df_raw.shape}")
print(f"Number of columns: {len(df_raw.columns)}")
print("\nFirst 10 columns:")
print(df_raw.columns[:10].tolist())

raw_label_counts = df_raw["Label"].value_counts().reset_index()
raw_label_counts.columns = ["label", "count"]
raw_label_counts.to_csv(CSV_DIR / "02_wednesday_raw_label_counts.csv", index=False)

display(raw_label_counts)

In [ ]:
# ============================================================
# CELL 8 — FILTER TO BENIGN + DOS-RELATED TRAFFIC
# ============================================================

DOS_LABELS = [
    "DoS Hulk",
    "DoS GoldenEye",
    "DoS slowloris",
    "DoS Slowhttptest",
]

KEEP_LABELS = ["BENIGN"] + DOS_LABELS

df_filtered = df_raw[df_raw["Label"].isin(KEEP_LABELS)].copy()
df_filtered["traffic_group"] = np.where(df_filtered["Label"] == "BENIGN", "BENIGN", "DoS_ATTACK")

filtered_label_counts = df_filtered["Label"].value_counts().reset_index()
filtered_label_counts.columns = ["label", "count"]

group_counts = df_filtered["traffic_group"].value_counts().reset_index()
group_counts.columns = ["traffic_group", "count"]

filtered_label_counts.to_csv(CSV_DIR / "03_filtered_label_counts.csv", index=False)
group_counts.to_csv(CSV_DIR / "04_filtered_group_counts.csv", index=False)

print(f"Filtered shape: {df_filtered.shape}")
display(filtered_label_counts)
display(group_counts)

In [ ]:
# ============================================================
# CELL 9 — QUICK DATA QUALITY CHECKS
# ============================================================

numeric_candidate_cols = [col for col in df_filtered.columns if col not in ["Label", "traffic_group"]]

inf_mask = np.isinf(df_filtered[numeric_candidate_cols].select_dtypes(include=[np.number]))
inf_count = int(inf_mask.sum().sum())

missing_summary = df_filtered.isna().sum().reset_index()
missing_summary.columns = ["column_name", "missing_count"]
missing_summary = missing_summary.sort_values("missing_count", ascending=False)

duplicate_count = int(df_filtered.duplicated().sum())

data_quality_summary = pd.DataFrame({
    "metric": [
        "n_rows_filtered",
        "n_columns_filtered",
        "n_numeric_candidate_columns",
        "total_missing_values",
        "total_infinite_values",
        "duplicate_rows",
    ],
    "value": [
        int(df_filtered.shape[0]),
        int(df_filtered.shape[1]),
        int(len(numeric_candidate_cols)),
        int(df_filtered.isna().sum().sum()),
        inf_count,
        duplicate_count,
    ],
})

data_quality_summary.to_csv(CSV_DIR / "05_data_quality_summary.csv", index=False)
missing_summary.to_csv(CSV_DIR / "06_missing_values_by_column.csv", index=False)

display(data_quality_summary)
display(missing_summary.head(10))

In [ ]:
# ============================================================
# CELL 10 — CLASS DISTRIBUTION VISUALISATIONS
# ============================================================

label_colors = [COLOR_REAL] + [COLOR_ATTACK] * max(0, len(filtered_label_counts) - 1)

plt.figure(figsize=(9, 5))
plt.bar(filtered_label_counts["label"], filtered_label_counts["count"], color=label_colors[:len(filtered_label_counts)])
plt.title("Filtered Label Distribution — Wednesday CICIDS")
plt.xlabel("Label")
plt.ylabel("Count")
plt.xticks(rotation=20, ha="right")
save_plot("01_filtered_label_distribution.png")
plt.show()

plt.figure(figsize=(7, 5))
plt.bar(group_counts["traffic_group"], group_counts["count"], color=[COLOR_REAL, COLOR_ATTACK])
plt.title("BENIGN vs DoS Group Distribution")
plt.xlabel("Traffic Group")
plt.ylabel("Count")
save_plot("02_benign_vs_dos_group_distribution.png")
plt.show()

In [ ]:
# ============================================================
# CELL 11 — CLEAN THE FEATURE MATRIX
# ============================================================

numeric_df = df_filtered.select_dtypes(include=[np.number]).copy()
numeric_df = numeric_df.replace([np.inf, -np.inf], np.nan)

columns_with_nan = numeric_df.columns[numeric_df.isna().any()].tolist()
print(f"Columns with NaN after replacing inf: {len(columns_with_nan)}")

clean_numeric_df = numeric_df.drop(columns=columns_with_nan)

nunique_per_col = clean_numeric_df.nunique()
zero_variance_cols = nunique_per_col[nunique_per_col <= 1].index.tolist()
clean_numeric_df = clean_numeric_df.drop(columns=zero_variance_cols)

feature_names = clean_numeric_df.columns.tolist()

feature_cleaning_summary = pd.DataFrame({
    "metric": [
        "initial_numeric_columns",
        "columns_removed_due_to_nan_or_inf",
        "columns_removed_due_to_zero_variance",
        "final_feature_count",
    ],
    "value": [
        int(numeric_df.shape[1]),
        int(len(columns_with_nan)),
        int(len(zero_variance_cols)),
        int(len(feature_names)),
    ],
})

feature_cleaning_summary.to_csv(CSV_DIR / "07_feature_cleaning_summary.csv", index=False)

print(f"Final feature count: {len(feature_names)}")
display(feature_cleaning_summary)
print("First 20 final features:")
print(feature_names[:20])

In [ ]:
# ============================================================
# CELL 12 — BUILD THE MODELLING DATASET
# ============================================================

model_df = clean_numeric_df.copy()
model_df["Label"] = df_filtered["Label"].values
model_df["traffic_group"] = df_filtered["traffic_group"].values

benign_df = model_df[model_df["traffic_group"] == "BENIGN"].copy()
dos_df = model_df[model_df["traffic_group"] == "DoS_ATTACK"].copy()

N_BENIGN_SAMPLE = 40000
N_DOS_SAMPLE = 40000

benign_sample = benign_df.sample(n=min(N_BENIGN_SAMPLE, len(benign_df)), random_state=SEED)
dos_sample = dos_df.sample(n=min(N_DOS_SAMPLE, len(dos_df)), random_state=SEED)

train_df = pd.concat([benign_sample, dos_sample], axis=0).sample(frac=1, random_state=SEED).reset_index(drop=True)

training_subset_summary = pd.DataFrame({
    "traffic_group": train_df["traffic_group"].value_counts().index,
    "count": train_df["traffic_group"].value_counts().values,
})

training_subset_summary.to_csv(CSV_DIR / "08_training_subset_group_counts.csv", index=False)

print(f"Training subset shape: {train_df.shape}")
display(training_subset_summary)

In [ ]:
# ============================================================
# CELL 13 — SCALE THE FEATURES AND BUILD THE DATALOADER
# ============================================================

X_train = train_df[feature_names].astype(np.float32).values

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)

print(f"Scaled feature matrix shape: {X_train_scaled.shape}")
print(f"Scaled mean (approx): {X_train_scaled.mean():.4f}")
print(f"Scaled std (approx): {X_train_scaled.std():.4f}")

BATCH_SIZE = 256

tensor_x = torch.tensor(X_train_scaled, dtype=torch.float32)
tensor_y = torch.zeros(len(X_train_scaled), dtype=torch.float32)

train_dataset = TensorDataset(tensor_x, tensor_y)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

scaling_summary = pd.DataFrame({
    "metric": ["n_rows_training", "n_features_training", "batch_size"],
    "value": [int(X_train_scaled.shape[0]), int(X_train_scaled.shape[1]), int(BATCH_SIZE)],
})
scaling_summary.to_csv(CSV_DIR / "09_scaling_and_loader_summary.csv", index=False)

display(scaling_summary)

In [ ]:
# ============================================================
# CELL 14 — TABULAR GAN MODEL DEFINITIONS
# ============================================================

FEATURE_DIM = X_train_scaled.shape[1]
LATENT_DIM = 32

class TabularGenerator(nn.Module):
    def __init__(self, latent_dim: int, output_dim: int):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(128),

            nn.Linear(128, 256),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(256),

            nn.Linear(256, 256),
            nn.LeakyReLU(0.2),

            nn.Linear(256, output_dim),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.model(z)

class TabularDiscriminator(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.2),

            nn.Linear(64, 1),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)

generator = TabularGenerator(latent_dim=LATENT_DIM, output_dim=FEATURE_DIM).to(DEVICE)
discriminator = TabularDiscriminator(input_dim=FEATURE_DIM).to(DEVICE)

print(generator)
print()
print(discriminator)

In [ ]:
# ============================================================
# CELL 15 — TRAINING CONFIGURATION
# ============================================================

NUM_EPOCHS = 300
LEARNING_RATE = 0.0002

loss_function = nn.BCELoss()
optimizer_discriminator = optim.Adam(discriminator.parameters(), lr=LEARNING_RATE, betas=(0.5, 0.999))
optimizer_generator = optim.Adam(generator.parameters(), lr=LEARNING_RATE, betas=(0.5, 0.999))

print(f"NUM_EPOCHS = {NUM_EPOCHS}")
print(f"LEARNING_RATE = {LEARNING_RATE}")

In [ ]:
# ============================================================
# CELL 16 — TRAIN THE TABULAR GAN
# ============================================================

history = []
start_time = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_d_loss = 0.0
    epoch_g_loss = 0.0
    batches = 0

    for real_samples, _ in train_loader:
        real_samples = real_samples.to(DEVICE)
        batch_size = real_samples.size(0)

        real_labels = torch.ones((batch_size, 1), device=DEVICE)
        fake_labels = torch.zeros((batch_size, 1), device=DEVICE)

        z = torch.randn((batch_size, LATENT_DIM), device=DEVICE)
        fake_samples = generator(z)

        all_samples = torch.cat((real_samples, fake_samples), dim=0)
        all_labels = torch.cat((real_labels, fake_labels), dim=0)

        discriminator.zero_grad()
        d_output = discriminator(all_samples)
        d_loss = loss_function(d_output, all_labels)
        d_loss.backward()
        optimizer_discriminator.step()

        z = torch.randn((batch_size, LATENT_DIM), device=DEVICE)
        generator.zero_grad()
        generated_samples = generator(z)
        d_generated = discriminator(generated_samples)

        g_loss = loss_function(d_generated, real_labels)
        g_loss.backward()
        optimizer_generator.step()

        epoch_d_loss += d_loss.item()
        epoch_g_loss += g_loss.item()
        batches += 1

    avg_d_loss = epoch_d_loss / batches
    avg_g_loss = epoch_g_loss / batches

    history.append({
        "epoch": epoch,
        "loss_discriminator": avg_d_loss,
        "loss_generator": avg_g_loss,
    })

    if epoch == 1 or epoch % 25 == 0 or epoch == NUM_EPOCHS:
        print(
            f"Epoch {epoch:3d}/{NUM_EPOCHS} | "
            f"Loss D: {avg_d_loss:.4f} | "
            f"Loss G: {avg_g_loss:.4f}"
        )

training_time_seconds = time.time() - start_time
history_df = pd.DataFrame(history)
history_df.to_csv(CSV_DIR / "10_training_loss_history.csv", index=False)

print(f"Training completed in {training_time_seconds:.2f} seconds.")

In [ ]:
# ============================================================
# CELL 17 — PLOT TRAINING LOSSES
# ============================================================

plt.figure(figsize=(9, 5))
plt.plot(history_df["epoch"], history_df["loss_discriminator"], label="Discriminator Loss", color=COLOR_REAL, linewidth=2)
plt.plot(history_df["epoch"], history_df["loss_generator"], label="Generator Loss", color=COLOR_ALT, linewidth=2)
plt.title("CICIDS Tabular GAN Training Losses")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
save_plot("03_cicids_tabular_gan_loss_curves.png")
plt.show()

In [ ]:
# ============================================================
# CELL 18 — GENERATE SYNTHETIC TRAFFIC SAMPLES
# ============================================================

def generate_synthetic_samples(generator: nn.Module, n_samples: int, latent_dim: int) -> np.ndarray:
    generator.eval()
    with torch.no_grad():
        z = torch.randn((n_samples, latent_dim), device=DEVICE)
        fake_scaled = generator(z).cpu().numpy()
    return fake_scaled

n_generate = len(X_train_scaled)
fake_scaled = generate_synthetic_samples(generator, n_samples=n_generate, latent_dim=LATENT_DIM)
fake_original_scale = scaler.inverse_transform(fake_scaled)

real_scaled = X_train_scaled.copy()
real_original_scale = scaler.inverse_transform(real_scaled)

print(f"Generated synthetic sample matrix shape: {fake_scaled.shape}")

In [ ]:
# ============================================================
# CELL 19 — SIMPLE QUANTITATIVE COMPARISON METRICS
# ============================================================

real_mean = real_scaled.mean(axis=0)
fake_mean = fake_scaled.mean(axis=0)

real_std = real_scaled.std(axis=0)
fake_std = fake_scaled.std(axis=0)

mean_l2_distance = float(np.linalg.norm(real_mean - fake_mean))
std_l2_distance = float(np.linalg.norm(real_std - fake_std))

sample_size = min(2000, len(real_scaled), len(fake_scaled))
sample_idx_real = np.random.choice(len(real_scaled), sample_size, replace=False)
sample_idx_fake = np.random.choice(len(fake_scaled), sample_size, replace=False)

pairwise_mean_distance = float(
    pairwise_distances(real_scaled[sample_idx_real], fake_scaled[sample_idx_fake]).mean()
)

quant_metrics = pd.DataFrame([{
    "mean_l2_distance": mean_l2_distance,
    "std_l2_distance": std_l2_distance,
    "mean_pairwise_distance_sampled": pairwise_mean_distance,
    "training_time_seconds": training_time_seconds,
    "n_generated_samples": int(n_generate),
    "n_features": int(FEATURE_DIM),
}])

quant_metrics.to_csv(CSV_DIR / "11_quantitative_alignment_metrics.csv", index=False)
display(quant_metrics)

In [ ]:
# ============================================================
# CELL 20 — PCA COMPARISON: REAL VS GENERATED
# ============================================================

combined_scaled = np.vstack([real_scaled, fake_scaled])

pca = PCA(n_components=2, random_state=SEED)
pca.fit(combined_scaled)

plot_sample_size = min(5000, len(real_scaled), len(fake_scaled))
real_plot_idx = np.random.choice(len(real_scaled), plot_sample_size, replace=False)
fake_plot_idx = np.random.choice(len(fake_scaled), plot_sample_size, replace=False)

pca_real = pca.transform(real_scaled[real_plot_idx])
pca_fake = pca.transform(fake_scaled[fake_plot_idx])

plt.figure(figsize=(8, 6))
plt.scatter(pca_real[:, 0], pca_real[:, 1], s=10, alpha=0.45, color=COLOR_REAL, label="Real")
plt.scatter(pca_fake[:, 0], pca_fake[:, 1], s=10, alpha=0.45, color=COLOR_FAKE, label="Generated")
plt.title("PCA Comparison — Real vs Generated CICIDS Traffic")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend()
save_plot("04_pca_real_vs_generated.png")
plt.show()

pca_variance = pd.DataFrame({
    "component": ["PC1", "PC2"],
    "explained_variance_ratio": pca.explained_variance_ratio_,
})
pca_variance.to_csv(CSV_DIR / "12_pca_explained_variance.csv", index=False)
display(pca_variance)

In [ ]:
# ============================================================
# CELL 21 — T-SNE COMPARISON: REAL VS GENERATED
# ============================================================

tsne_sample_size = min(2000, len(real_scaled), len(fake_scaled))
real_tsne_idx = np.random.choice(len(real_scaled), tsne_sample_size, replace=False)
fake_tsne_idx = np.random.choice(len(fake_scaled), tsne_sample_size, replace=False)

tsne_input = np.vstack([
    real_scaled[real_tsne_idx],
    fake_scaled[fake_tsne_idx],
])

tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate="auto",
    init="pca",
    random_state=SEED,
)
tsne_embedding = tsne.fit_transform(tsne_input)

tsne_real = tsne_embedding[:tsne_sample_size]
tsne_fake = tsne_embedding[tsne_sample_size:]

plt.figure(figsize=(8, 6))
plt.scatter(tsne_real[:, 0], tsne_real[:, 1], s=12, alpha=0.45, color=COLOR_REAL, label="Real")
plt.scatter(tsne_fake[:, 0], tsne_fake[:, 1], s=12, alpha=0.45, color=COLOR_FAKE, label="Generated")
plt.title("t-SNE Comparison — Real vs Generated CICIDS Traffic")
plt.xlabel("t-SNE Dimension 1")
plt.ylabel("t-SNE Dimension 2")
plt.legend()
save_plot("05_tsne_real_vs_generated.png")
plt.show()

In [ ]:
# ============================================================
# CELL 22 — FEATURE-DISTRIBUTION COMPARISON PLOTS
# ============================================================

selected_features = feature_names[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, feature in zip(axes, selected_features):
    feature_idx = feature_names.index(feature)

    real_vals = real_original_scale[:, feature_idx]
    fake_vals = fake_original_scale[:, feature_idx]

    lower = np.percentile(real_vals, 1)
    upper = np.percentile(real_vals, 99)

    ax.hist(np.clip(real_vals, lower, upper), bins=40, alpha=0.55, label="Real", color=COLOR_REAL, density=True)
    ax.hist(np.clip(fake_vals, lower, upper), bins=40, alpha=0.55, label="Generated", color=COLOR_FAKE, density=True)
    ax.set_title(feature)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2)
save_plot("06_selected_feature_distribution_comparisons.png")
plt.show()

In [ ]:
# ============================================================
# CELL 23 — REPORT-READY SUMMARY TABLE
# ============================================================

report_summary = pd.DataFrame({
    "item": [
        "multiday_files_explored",
        "wednesday_raw_rows",
        "wednesday_filtered_rows_benign_plus_dos",
        "final_feature_count",
        "training_subset_rows",
        "generated_rows",
        "mean_l2_distance",
        "std_l2_distance",
        "mean_pairwise_distance_sampled",
        "training_time_seconds",
    ],
    "value": [
        int(len(DATA_FILES)),
        int(df_raw.shape[0]),
        int(df_filtered.shape[0]),
        int(FEATURE_DIM),
        int(train_df.shape[0]),
        int(n_generate),
        round(mean_l2_distance, 4),
        round(std_l2_distance, 4),
        round(pairwise_mean_distance, 4),
        round(training_time_seconds, 2),
    ],
})

report_summary.to_csv(CSV_DIR / "13_report_ready_summary_table.csv", index=False)
display(report_summary)

## Interpretation guide for the report

Use the outputs from this notebook to discuss the following:

### 1. Multi-day exploration
A preliminary analysis across Monday, Tuesday, Wednesday, Thursday, and Friday shows that attack composition varies substantially by day.  
This supports the decision to **train on Wednesday only**, because it provides the most relevant BENIGN + DoS configuration for the assignment.

### 2. Why an MLP GAN was used
The CICIDS task involves **tabular network traffic features**, not images.  
A convolutional GAN would impose an artificial spatial structure, so an **MLP-based GAN** is the more appropriate architecture.

### 3. Why scaling was essential
CICIDS features operate on very different numeric ranges.  
Standardisation prevents a few large-scale variables from dominating training and usually improves adversarial stability.

### 4. Why a balanced training subset was used
The Wednesday file is heavily imbalanced, especially between BENIGN and specific attack categories.  
Using a balanced subset makes training more practical in Colab and reduces the risk that the generator simply reproduces the majority region of feature space.

### 5. How to interpret PCA and t-SNE
- Strong overlap suggests the synthetic samples broadly occupy similar regions of feature space.
- Clear separation suggests poor alignment between generated and real traffic.
- Partial overlap usually indicates that the GAN learned some structure but not the full data distribution.

### 6. Likely limitations
- GANs often struggle to preserve complex feature dependencies in tabular data.
- Good visual overlap in PCA or t-SNE does not guarantee perfect feature-level realism.
- Marginal distributions may look reasonable even when joint relationships remain imperfect.

In [ ]:
# ============================================================
# CELL 24 — OPTIONAL LIST OF SAVED OUTPUT FILES
# ============================================================

print("Saved CSV files:")
for path in sorted(CSV_DIR.glob("*.csv")):
    print("-", path)

print("\nSaved plot files:")
for path in sorted(PLOTS_DIR.glob("*.png")):
    print("-", path)